# Ensemble baselines
## Optical flow + Bounding box dynamics

In [1]:
import sys
import pandas as pd
import numpy as np
import torch
import ruptures
import cv2

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("OpenCV:", cv2.__version__)

Python: c:\Users\LOQ\Desktop\Senior 2\cctv-accident-ITS-system\ACCIDENT\baselines\heuristic\.venv\Scripts\python.exe
Pandas: 3.0.0
NumPy: 2.4.1
PyTorch: 2.10.0+cu128
CUDA available: True
OpenCV: 4.13.0


In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

from metrics import print_temporal_accuracy, print_spatial_accuracy
# Define paths and load video files
DATASET_PATH = Path("../../dataset")

%reload_ext autoreload
%autoreload 2

In [13]:

from pathlib import Path
import pandas as pd
import io
from contextlib import redirect_stdout
from metrics import print_temporal_accuracy, print_spatial_accuracy

HEURISTIC_PATH = Path(
    r"C:\Users\LOQ\Desktop\Senior 2\cctv-accident-ITS-system\ACCIDENT\baselines\heuristic"
)

ENSEMBLE_PATH = HEURISTIC_PATH / "output_ensemble_test.csv"
LOG_PATH = HEURISTIC_PATH / "output_ensemble_test_terminal_output.txt"

ensemble = pd.read_csv(ENSEMBLE_PATH)
true_df = true_df.copy()

def normalize_filename(value):
    return Path(str(value).replace("\\", "/")).name.strip().casefold()

ensemble["_match_key"] = ensemble["path"].map(normalize_filename)
true_df["_match_key"] = true_df["path"].map(normalize_filename)

if ensemble["_match_key"].duplicated().any():
    raise ValueError("Duplicate filenames found in ensemble predictions.")

if true_df["_match_key"].duplicated().any():
    raise ValueError("Duplicate filenames found in ground-truth annotations.")

matched = true_df.merge(
    ensemble,
    on="_match_key",
    how="inner",
    suffixes=("_true", "_pred"),
    validate="one_to_one"
)

if matched.empty:
    raise ValueError("No matching filenames found.")

matched_predictions = ensemble[
    ensemble["_match_key"].isin(matched["_match_key"])
].copy()

matched_truth = true_df[
    true_df["_match_key"].isin(matched["_match_key"])
].copy()

matched_predictions["path"] = matched_predictions["_match_key"]
matched_truth["path"] = matched_truth["_match_key"]

if len(matched) != len(ensemble):
    raise ValueError(
        f"Only {len(matched)} of {len(ensemble)} predictions matched ground truth."
    )

buffer = io.StringIO()

with redirect_stdout(buffer):
    print("ENSEMBLE TEST EVALUATION")
    print("========================")
    print("Prediction file:", ENSEMBLE_PATH)
    print("Ground-truth rows available:", len(true_df))
    print("Ensemble predictions:", len(ensemble))
    print("Matched videos:", len(matched))

    print("\nTEMPORAL ACCURACY")
    print("-----------------")
    print_temporal_accuracy(
        predictions=matched_predictions,
        true_df=matched_truth,
        sigmas=[0.5, 1, 2]
    )

    print("\nSPATIAL ACCURACY")
    print("----------------")
    print_spatial_accuracy(
        predictions=matched_predictions,
        true_df=matched_truth,
        sigmas=[0.5, 1, 2]
    )

results = buffer.getvalue()
print(results)

LOG_PATH.write_text(results, encoding="utf-8")
print("Evaluation log saved to:", LOG_PATH)


ENSEMBLE TEST EVALUATION
Prediction file: C:\Users\LOQ\Desktop\Senior 2\cctv-accident-ITS-system\ACCIDENT\baselines\heuristic\output_ensemble_test.csv
Ground-truth rows available: 2027
Ensemble predictions: 400
Matched videos: 400

TEMPORAL ACCURACY
-----------------
Temporal task:
____________________
| Sigma | Accuracy |
| 0.50  | 0.166    |
| 1.00  | 0.287    |
| 2.00  | 0.487    |



SPATIAL ACCURACY
----------------
Spatial task:
Normalized sigmas:  0.13087649006699523 0.1226289247383957
____________________
| Sigma | Accuracy |
| 0.50  | 0.121    |
| 1.00  | 0.281    |
| 2.00  | 0.549    |



Evaluation log saved to: C:\Users\LOQ\Desktop\Senior 2\cctv-accident-ITS-system\ACCIDENT\baselines\heuristic\output_ensemble_test_terminal_output.txt


In [14]:

RESULTS_DIR = HEURISTIC_PATH.parents[2] / "results" / "heuristic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = RESULTS_DIR / "output_ensemble_test.csv"

results_df = pd.DataFrame([
    {
        "method": "ensemble",
        "task": "temporal",
        "sigma_0.5": 0.166,
        "sigma_1.0": 0.287,
        "sigma_2.0": 0.487,
        "matched_videos": 400
    },
    {
        "method": "ensemble",
        "task": "spatial",
        "sigma_0.5": 0.121,
        "sigma_1.0": 0.281,
        "sigma_2.0": 0.549,
        "matched_videos": 400
    }
])

results_df.to_csv(CSV_PATH, index=False)

print("CSV results saved to:", CSV_PATH)


CSV results saved to: C:\Users\LOQ\Desktop\Senior 2\cctv-accident-ITS-system\results\heuristic\output_ensemble_test.csv
